[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/S12_01_series_temporales.ipynb)

# S12 · 01 · Series temporales: baseline, descomposición y ARIMA/SARIMAX

**Tiempo estimado:** 1 h · **Sprint 12** (UD 4.7) · Cliente: **TurisData Canarias**

**Lo que vas a aprender**
1. Por qué una serie temporal **no se parte al azar** (train/test por fecha).
2. Qué es un **baseline** (ingenuo y estacional) y por qué es el rival a batir.
3. Descomponer una serie en **tendencia, estacionalidad y resto**.
4. Ajustar un modelo **ARIMA/SARIMAX** con `statsmodels` y evaluarlo con una partición temporal.

**Contexto.** La dirección de TurisData quiere saber cuánta ocupación tendrá el alojamiento
la próxima semana para organizar personal y compras. Tenemos la ocupación diaria (%) de 2023 a 2025.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

ocup = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"]).set_index("fecha")
ocup.index.freq = "D"
print(ocup.shape)
ocup.head()

## 1. Mirar la serie
**Idea.** Antes de modelar, se dibuja. Buscamos tres cosas: *tendencia* (¿sube o baja a largo plazo?),
*estacionalidad* (patrones que se repiten: la semana, el año) y *ruido*.

In [ ]:
fig, ax = plt.subplots()
ocup["ocupacion_pct"].plot(ax=ax, lw=0.8)
ax.set_title("Ocupación diaria del alojamiento (2023-2025)")
ax.set_xlabel("Fecha"); ax.set_ylabel("Ocupación (%)")
plt.show()

# Patrón semanal y anual, promediando
por_dia = ocup["ocupacion_pct"].groupby(ocup.index.dayofweek).mean()
por_mes = ocup["ocupacion_pct"].groupby(ocup.index.month).mean()
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
por_dia.index = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
por_dia.plot.bar(ax=ejes[0], color="#3b7ea1"); ejes[0].set_title("Ocupación media por día de la semana")
por_mes.plot.bar(ax=ejes[1], color="#e08e45"); ejes[1].set_title("Ocupación media por mes")
ejes[0].set_ylabel("Ocupación (%)"); ejes[1].set_xlabel("Mes")
plt.tight_layout(); plt.show()

**Ejercicio 1.** ¿Qué día de la semana tiene más ocupación media y qué mes tiene menos?
Guarda el nombre del día en `dia_max` (texto, p. ej. `"sáb"`) y el número de mes en `mes_min` (entero de 1 a 12).

In [ ]:
# TODO: usa idxmax() sobre por_dia y idxmin() sobre por_mes
...
assert dia_max in ("vie", "sáb", "dom"), "Revisa dia_max: el fin de semana es el que más ocupa."
assert mes_min == 9, "Revisa mes_min: mira las barras de ocupación por mes y quédate con la más baja."

## 2. Partición temporal (¡sin barajar!)
**Idea.** Con datos de tabla partimos al azar. Con series **no**: el modelo no puede ver el futuro.
Entrenamos con el pasado y evaluamos con los últimos días. Aquí reservamos los **últimos 91 días** (13 semanas)
como test. Es la misma idea de "sin fuga de datos" que ya conoces, aplicada al tiempo.

In [ ]:
DIAS_TEST = 91
serie = ocup["ocupacion_pct"]
entrenamiento, prueba = serie.iloc[:-DIAS_TEST], serie.iloc[-DIAS_TEST:]
print("Entrenamiento:", entrenamiento.index.min().date(), "→", entrenamiento.index.max().date(), len(entrenamiento))
print("Prueba       :", prueba.index.min().date(), "→", prueba.index.max().date(), len(prueba))

## 3. Baselines: el rival que hay que batir
**Idea.** Un modelo complejo solo vale la pena si gana a algo muy simple.
- **Ingenuo**: mañana será igual que hoy (repetimos el último valor conocido).
- **Estacional ingenuo**: el lunes que viene será como el último lunes (repetimos la última semana).

**Cómo evaluamos.** Imitamos el uso real: cada domingo predecimos los **7 días siguientes** con los datos
conocidos hasta ese momento. Repetimos en las 13 semanas del test y medimos el **error absoluto medio (MAE)**,
en puntos de ocupación.

In [ ]:
HORIZONTE = 7
origenes = list(range(len(entrenamiento), len(serie), HORIZONTE))   # posiciones donde empieza cada semana de test
origenes = [o for o in origenes if o + HORIZONTE <= len(serie)]

def mae(real, pred):
    return float(np.mean(np.abs(np.asarray(real) - np.asarray(pred))))

def evaluar_semanal(predictor):
    """predictor(historia: Series, origen: int) -> array de 7 valores. Devuelve (MAE, predicciones concatenadas)."""
    preds, reales = [], []
    for o in origenes:
        preds.append(np.asarray(predictor(serie.iloc[:o], o)))
        reales.append(serie.iloc[o:o + HORIZONTE].values)
    preds, reales = np.concatenate(preds), np.concatenate(reales)
    return mae(reales, preds), preds

def pred_ingenuo(historia, o):
    return np.repeat(historia.iloc[-1], HORIZONTE)

def pred_estacional(historia, o):
    return historia.iloc[-7:].values

mae_ing, p_ing = evaluar_semanal(pred_ingenuo)
mae_est, p_est = evaluar_semanal(pred_estacional)
print(f"MAE ingenuo    : {mae_ing:.2f} puntos")
print(f"MAE estacional : {mae_est:.2f} puntos")

**Ejercicio 2.** Un tercer baseline: la **media de las últimas 4 semanas** para cada día de la semana
(más estable que copiar solo la última semana). Completa `pred_media4`: debe devolver 7 valores, uno por día,
promediando `historia.iloc[-7]`, `historia.iloc[-14]`, `historia.iloc[-21]` y `historia.iloc[-28]`, y así para
cada día de la semana siguiente. Después se calcula `mae_media4`.

In [ ]:
def pred_media4(historia, o):
    # TODO: apila las últimas 28 observaciones en una matriz de 4 filas x 7 columnas y haz la media por columnas
    ...

mae_media4, p_media4 = evaluar_semanal(pred_media4)
print(f"MAE media 4 semanas: {mae_media4:.2f}")
assert len(pred_media4(serie.iloc[:400], 400)) == 7, "pred_media4 debe devolver 7 valores."
assert mae_media4 < mae_ing, "La media de 4 semanas debería batir al ingenuo; revisa el reshape (4, 7) y el eje de la media."

## 4. Descomposición: tendencia + estacionalidad + resto
**Idea.** Una serie se puede ver como suma de piezas: `serie = tendencia + estacionalidad + resto`.
`seasonal_decompose` estima la parte que se repite cada 7 días y una tendencia suavizada.
Lo que queda (el *resto*) es lo que ningún patrón explica. Usamos solo el entrenamiento.

In [ ]:
desc = seasonal_decompose(entrenamiento, model="additive", period=7)
fig, ejes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, (nombre, s) in zip(ejes, [("Observada", desc.observed), ("Tendencia", desc.trend),
                                   ("Estacionalidad semanal", desc.seasonal), ("Resto", desc.resid)]):
    ax.plot(s, lw=0.8); ax.set_ylabel(nombre)
ejes[0].set_title("Descomposición aditiva (periodo = 7 días)")
plt.tight_layout(); plt.show()
print("Efecto semanal estimado (puntos sobre/bajo la media):")
print(desc.seasonal.iloc[:7].round(1).to_string())

**Ejercicio 3.** ¿Cuánta de la variación se explica por la estacionalidad semanal? Calcula en `amplitud_semanal`
la diferencia entre el mayor y el menor efecto estacional (`desc.seasonal.max() - desc.seasonal.min()`),
redondeada a 1 decimal, y guarda en `resto_std` la desviación típica del resto (sin NaN).

In [ ]:
# TODO: max - min del componente seasonal; std() ignora NaN en pandas si usas dropna()
...
assert 5 < amplitud_semanal < 15, "amplitud_semanal debería rondar la brecha entre fin de semana y entre semana (~9 puntos)."
assert 0 < resto_std < serie.std(), "resto_std debe ser menor que la desviación de la serie original."

## 5. ARIMA y SARIMAX
**Idea sin fórmulas.**
- **AR (autorregresivo)**: el valor de hoy depende de los de días anteriores.
- **I (integrado)**: se trabaja con *diferencias* para quitar tendencias.
- **MA (media móvil)**: el valor depende de los errores recientes del modelo.
- **S (estacional)**: lo mismo pero mirando "hace 7 días".
- **X (exógenas)**: variables externas que conocemos de antemano, como los **festivos**.

Como la estacionalidad anual (invierno alto, verano bajo) es larga para SARIMAX, la representamos con
dos ondas seno/coseno del año (**términos de Fourier**), que son calendario y por tanto conocidos en el futuro.
**Cuidado:** no usamos la temperatura como variable, porque no la conoceríamos con antelación (sería *fuga*).

In [ ]:
def exogenas(indice):
    dia_anio = indice.dayofyear.values
    X = pd.DataFrame(index=indice)
    for k in (1, 2):
        X[f"sin{k}"] = np.sin(2 * np.pi * k * dia_anio / 365.25)
        X[f"cos{k}"] = np.cos(2 * np.pi * k * dia_anio / 365.25)
    X["festivo"] = ocup.loc[indice, "festivo"].values
    return X

X_todo = exogenas(serie.index)
X_ent = X_todo.iloc[:-DIAS_TEST]

modelo = SARIMAX(entrenamiento, exog=X_ent, order=(1, 0, 1), seasonal_order=(1, 1, 1, 7),
                 enforce_stationarity=False, enforce_invertibility=False)
res = modelo.fit(disp=False, maxiter=200)
print(res.summary().tables[1])

**Cómo usarlo semana a semana.** No reajustamos el modelo cada semana (sería lento): reutilizamos los
parámetros aprendidos y solo le damos la historia nueva con `res.apply(...)`. Es el mismo proceso que en producción:
cada domingo se actualiza la historia y se pide la previsión de 7 días.

In [ ]:
def pred_sarimax(historia, o):
    r = res.apply(historia, exog=X_todo.iloc[:o], refit=False)
    return r.forecast(HORIZONTE, exog=X_todo.iloc[o:o + HORIZONTE]).values

mae_sar, p_sar = evaluar_semanal(pred_sarimax)
print(f"MAE SARIMAX: {mae_sar:.2f}   (estacional ingenuo: {mae_est:.2f}, media 4 sem.: {mae_media4:.2f})")

**Ejercicio 4.** Prueba una segunda especificación **sin** los festivos ni Fourier (solo `order=(1,0,1)`, `seasonal_order=(1,1,1,7)`)
y compara. Guarda su MAE en `mae_sin_exog`. Pista: `SARIMAX(entrenamiento, order=..., seasonal_order=...)`,
y en `res2.apply(historia, refit=False)` no hay `exog`.

In [ ]:
# TODO: ajusta res2 sin exog y repite el bucle semanal con res2.apply(historia, refit=False).forecast(HORIZONTE)
...
print(f"MAE SARIMAX sin exógenas: {mae_sin_exog:.2f}  |  con exógenas: {mae_sar:.2f}")
assert 0 < mae_sin_exog < 15, "mae_sin_exog debe ser un MAE razonable (en puntos de ocupación)."

## 6. Comparar y mirar los errores
Una tabla y un gráfico: ¿gana el modelo sofisticado a los baselines? Si la ganancia es pequeña, a veces
**el baseline es la respuesta correcta para el cliente** (más simple, más fácil de explicar).

In [ ]:
comparativa = pd.Series({"Ingenuo": mae_ing, "Estacional (t-7)": mae_est, "Media 4 semanas": mae_media4,
                         "SARIMAX sin exógenas": mae_sin_exog, "SARIMAX + festivos + Fourier": mae_sar}).sort_values()
print(comparativa.round(2).to_string())

fig, ax = plt.subplots()
ax.plot(prueba.index, prueba.values, label="Real", color="black", lw=1.2)
n = len(p_sar); ind = serie.index[origenes[0]:origenes[0] + n]
ax.plot(ind, p_est, label="Estacional t-7", alpha=.8)
ax.plot(ind, p_sar, label="SARIMAX", alpha=.9)
ax.set_title("Previsión a 7 días en las últimas 13 semanas"); ax.set_ylabel("Ocupación (%)"); ax.legend()
plt.show()

## 7. Mini-reto integrador
La dirección pide un **número de control**: ¿cuánto mejora (en %) el mejor modelo respecto al mejor baseline?
Guarda en `mejor_baseline` el menor MAE entre los tres baselines y en `mejora_pct` el porcentaje de mejora
de `mae_sar` respecto a él (`100 * (baseline - modelo) / baseline`, redondeado a 1 decimal; puede ser negativo).
Escribe además en `conclusion` una frase (texto) para el cliente.

In [ ]:
# TODO: min de los tres MAE de baseline; la fórmula de mejora está en el enunciado
...
print(conclusion)
assert abs(mejor_baseline - min(mae_ing, mae_est, mae_media4)) < 1e-9, "mejor_baseline es el mínimo de los tres MAE de baseline."
assert isinstance(conclusion, str) and len(conclusion) > 20, "Escribe una conclusión en una frase para el cliente."

## Preguntas de reflexión
1. ¿Por qué barajar las filas antes de partir train/test sería un error en una serie temporal?
2. Si el mejor baseline queda muy cerca de SARIMAX, ¿qué recomendarías al cliente y por qué?
3. ¿Qué otra variable exógena podría ayudar y **cuándo** la conoceríamos? (pista: ¿se conoce el futuro?)
4. El modelo prevé bien el día "normal"; ¿qué pasaría con un evento extraordinario (un temporal, un congreso)?

## Resumen: qué has aprendido
- Las series se parten **por fecha** y se evalúan imitando el uso real (previsión a 7 días).
- Siempre hay que compararse con un **baseline**; a veces gana o casi.
- La descomposición separa tendencia, estacionalidad y resto; SARIMAX combina memoria, estacionalidad y variables conocidas.
- Evita usar como entrada datos que no tendrás en el momento de predecir.